In [1]:
import pandas as pd

In [34]:
text = """
I can't believe this product is AMAZING!!! 😍 I've used it 3 times,
and it doesn’t disappoint. The build-quality is top-notch,
but it's kinda expensive... Would I buy it again? Yes, absolutely!!
"""

In [26]:
df = pd.read_csv("../../data/bbc-news-data.csv", sep='\t')
df.head()

,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...


In [27]:
df.shape

(2225, 4)

In [28]:
df['text'] = df['title']+" "+df['content']

In [29]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer


In [ ]:
import re
import contractions
import spacy
from nltk import pos_tag

In [ ]:
#tagged_tokens = pos_tag(tokens)

In [31]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [32]:
def preprocess(text):
    text = text.lower()
    text = contractions.fix(text)
    text = re.sub(r'\W+', ' ', text)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)

In [ ]:
from nltk.stem import PorterStemmer, SnowballStemmer

porter = PorterStemmer()
snowball = SnowballStemmer("english")

def stemming(tokens):
    porter_stems = [porter.stem(word) for word in tokens]
    snowball_stems = [snowball.stem(word) for word in tokens]


In [33]:
df['clean_text'] = df['text'].apply(preprocess)
df['clean_text'].head()

0    ad sale boost time warner profit quarterly pro...
1    dollar gain greenspan speech dollar hit highes...
2    yukos unit buyer face loan claim owner embattl...
3    high fuel price hit ba profit british airway b...
4    pernod takeover talk lift domecq share uk drin...
Name: clean_text, dtype: str

In [35]:
nlp = spacy.load("en_core_web_sm")

doc = nlp(text)

In [42]:
def spacy_lemmatization(text):
    doc = nlp(text)
    lemmatized = [
        token.lemma_ for token in doc
        if not token.is_stop and not token.is_punct
    ]
    return lemmatized

In [16]:
raw_vocab = set(" ".join(df['text']).split())
clean_vocab = set(" ".join(df['clean_text']).split())

print("Raw Vocabulary Size:", len(raw_vocab))
print("Clean Vocabulary Size:", len(clean_vocab))

Raw Vocabulary Size: 65553
Clean Vocabulary Size: 31493


In [19]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_tfidf = vectorizer.fit_transform(df['clean_text'])
X_tfidf.shape

(2225, 5000)

In [45]:
#bow

from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer()


x_bow = count_vectorizer.fit_transform(df['clean_text'])
print(x_bow.toarray())

#count_vectorizer.vocabulary_

[[0 1 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]]


In [55]:
documents = df['clean_text'].astype(str).apply(lambda x: x.split()).tolist()

In [61]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=documents,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1
)

In [62]:
vector = model.wv["product"]
print(vector.shape)  
print(vector[:10]) 

(100,)
[-0.33386326  0.13652211  0.34162888  0.10526516 -0.08192644 -0.36948982
  0.42578787  0.2998116  -0.27837697 -0.58288807]


In [63]:
model.wv.most_similar("amazing")

[('hell', 0.9874151349067688),
 ('sad', 0.9848983883857727),
 ('luck', 0.9818481206893921),
 ('tired', 0.9797980785369873),
 ('unfortunately', 0.979789137840271),
 ('comfortable', 0.97963547706604),
 ('realised', 0.9794918298721313),
 ('obsessed', 0.9790973663330078),
 ('impress', 0.9788680672645569),
 ('anyway', 0.9784827828407288)]